# Agent BlackBox Arena Training Rerun Guide

This notebook is a rerun guide and evidence reader for the OpenEnv Hackathon submission. The final full training run was executed through Hugging Face Jobs on H200 hardware, not inside this notebook. The notebook shows how to install the project, run smoke checks, inspect the SFT/GRPO training path, and review the real final summaries, plots, and logs.

It does not fake outputs, does not require an HF token for smoke mode, and does not require H200 hardware to understand the project.

## Project Summary

Agent BlackBox Arena is an OpenEnv-style repair environment for agent reliability CI. Observability can show what happened in an agent trace; this project trains an agent to decide what should change next.

The environment turns failed synthetic traces into repair episodes. The model learns to inspect evidence, replay the incident, identify a root cause, propose a bounded patch, run visible and hidden regressions, preserve valid behavior, and generate a bounded repair certificate.

In [ ]:
## Environment Loop

```text
failed trace -> evidence -> root cause -> repair patch -> visible replay -> hidden regressions -> certificate
```

The repair certificate is only emitted after the repair chain passes verifier checks. Hidden regression variants are not exposed in the observation.


# Setup
# If running from a fresh notebook runtime, uncomment the clone command first.
# !git clone https://github.com/Kenxpx/Agent-Blackbox-Arena.git
# %cd Agent-Blackbox-Arena

!pip install -e ".[training]"

In [ ]:
# Smoke checks
# These run locally/CPU and do not require a Hugging Face token or H200.
!python scripts/self_check.py
!python scripts/space_smoke.py
!python -m pytest


## Dataset And Training Path

The training path uses an SFT warmstart followed by GRPO-style verifier reward optimization. The SFT stage teaches strict JSON/action format. GRPO then optimizes against deterministic repair rewards from the verifier.

The final evaluation covers three prompt variants:

- `standard`
- `shuffled_surface_blind`
- `combined_blind_shuffle`

The smoke commands below are intentionally tiny. They verify the parser, verifier reward, dataset generation, and training/evaluation scaffolds without running the final H200 job.

In [ ]:
# Tiny rerun commands
!python training/train_sft_warmstart.py --smoke --output-dir outputs/sft_smoke
!python training/train_json_grpo.py --smoke --output-dir outputs/grpo_smoke
!python training/evaluate_model.py --smoke --output-dir outputs/eval_smoke


## Full Training Command Reference

The final selected model was trained outside this notebook with Hugging Face Jobs on H200 hardware:

- Model: `Qwen/Qwen3-4B-Instruct-2507`
- Result: `SFT+GRPO final H200`
- HF Job: `69edcef7d70108f37acdfeb3`
- Output root: `outputs/larger_models/qwen3_4b_2507_final_h200/`

The relevant training code is in `training/train_sft_warmstart.py`, `training/train_json_grpo.py`, and the larger-model run scripts under `scripts/`. Full H200 reruns require GPU/HF Jobs resources and are not required for this notebook smoke rerun.

In [ ]:
# Final result table from the checked-in metrics asset.
import json
from pathlib import Path

metrics_path = Path("docs/final_assets/metrics/final_qwen3_4b_metrics.json")
if metrics_path.exists():
    data = json.loads(metrics_path.read_text())
    print(data["metadata"])
    for row in data["variants"]:
        print({
            "variant": row["variant"],
            "overall": f"{row['overall_score']:.4f}",
            "cert": f"{row['certificate_success_rate']:.4f}",
            "evidence": f"{row['evidence_correct_rate']:.4f}",
            "hidden": f"{row['hidden_regression_pass_rate']:.4f}",
            "valid": f"{row['valid_preservation_rate']:.4f}",
            "invalid_json": f"{row['invalid_json_rate']:.4f}",
        })
else:
    print(f"Missing {metrics_path}; see SUBMISSION_EVIDENCE.md for final metrics.")


# Plot display: shows checked-in final assets when available.
from IPython.display import Image, Markdown, display
from pathlib import Path

plot_paths = [
    Path("docs/final_assets/plots/qwen3_4b_sft_loss.png"),
    Path("docs/final_assets/plots/qwen3_4b_grpo_reward.png"),
    Path("docs/final_assets/plots/qwen3_4b_eval_overall.png"),
    Path("docs/final_assets/plots/qwen3_4b_certificate_success.png"),
    Path("docs/final_assets/plots/qwen3_4b_evidence_correctness.png"),
    Path("docs/final_assets/plots/qwen3_4b_safety_rates.png"),
]

for path in plot_paths:
    if path.exists():
        display(Markdown(f"### `{path.as_posix()}`"))
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"Missing plot locally: `{path.as_posix()}`"))

## Logs And Evidence Links

- Full HF job log: `logs/final/hf_job_qwen3_4b_final_h200_69edcef7d70108f37acdfeb3_tail.txt`
- Evidence ledger: `SUBMISSION_EVIDENCE.md`
- Final audit: `FINAL_SUBMISSION_AUDIT.md`
- Recent H200 report: `FINAL_RECENT_H200_RUNS_REPORT.md`
- Form checklist: `FINAL_FORM_SUBMISSION_CHECKLIST.md`
- Final assets: `docs/final_assets/`

The log and assets are included for auditability. The notebook does not embed huge logs.

## Claim Boundaries

Allowed claim: `Qwen/Qwen3-4B-Instruct-2507 SFT+GRPO final H200` is the selected trained result for the reported synthetic MVP families, prompt variants, and eval seeds `1000-1019`.

Not claimed:

- global safety
- production safety
- SOTA
- unlimited generalization
- Qwen2.5-3B success
- 1.5B success
- failed/error H200 jobs as successful
- GRPO learning the task from scratch without the SFT warmstart and challenge curriculum

The Agent Repair Certificate is bounded to synthetic incident families, prompt variants, hidden regressions, and the verifier horizon.